# Day 41 — NLP & Text Processing
### Tokenisation · Stop Words · Stemming · TF-IDF · Text Classification

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('dark_background')

# download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("All imports ready! ✅")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vijen\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vijen\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vijen\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vijen\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vijen\AppData\Roaming\nltk_data...


All imports ready! ✅


## 2. Tokenisation

In [2]:
print("=" * 55)
print("       TOKENISATION")
print("=" * 55)

text = """Natural Language Processing is fascinating. 
It helps computers understand human language. 
NLP powers chatbots, translators and search engines!"""

print("ORIGINAL TEXT:")
print(text)

# sentence tokenisation
sentences = sent_tokenize(text)
print(f"\nSENTENCE TOKENISATION ({len(sentences)} sentences):")
for i, s in enumerate(sentences):
    print(f"  {i+1}: {s}")

# word tokenisation
words = word_tokenize(text)
print(f"\nWORD TOKENISATION ({len(words)} tokens):")
print(f"  {words}")

# simple split vs NLTK
print(f"\nSimple split(): {text.split()[:8]} ...")
print(f"NLTK tokenize: {word_tokenize(text)[:8]} ...")
print(f"\nDifference: NLTK handles punctuation separately!")
print(f"  'fascinating.' → NLTK splits to ['fascinating', '.']")

       TOKENISATION
ORIGINAL TEXT:
Natural Language Processing is fascinating. 
It helps computers understand human language. 
NLP powers chatbots, translators and search engines!

SENTENCE TOKENISATION (3 sentences):
  1: Natural Language Processing is fascinating.
  2: It helps computers understand human language.
  3: NLP powers chatbots, translators and search engines!

WORD TOKENISATION (22 tokens):
  ['Natural', 'Language', 'Processing', 'is', 'fascinating', '.', 'It', 'helps', 'computers', 'understand', 'human', 'language', '.', 'NLP', 'powers', 'chatbots', ',', 'translators', 'and', 'search', 'engines', '!']

Simple split(): ['Natural', 'Language', 'Processing', 'is', 'fascinating.', 'It', 'helps', 'computers'] ...
NLTK tokenize: ['Natural', 'Language', 'Processing', 'is', 'fascinating', '.', 'It', 'helps'] ...

Difference: NLTK handles punctuation separately!
  'fascinating.' → NLTK splits to ['fascinating', '.']


## 3. Stop Words & Text Cleaning

In [3]:
print("=" * 55)
print("       STOP WORDS & TEXT CLEANING")
print("=" * 55)

stop_words = set(stopwords.words('english'))
print(f"Total English stop words: {len(stop_words)}")
print(f"Examples: {sorted(list(stop_words))[:15]}")

# clean + remove stop words
def clean_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # tokenise
    tokens = word_tokenize(text)
    # remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

sample = "The movie was not very good but the acting was absolutely brilliant!"
cleaned = clean_text(sample)

print(f"\nORIGINAL:  '{sample}'")
print(f"CLEANED:    {cleaned}")
print(f"\nRemoved {len(word_tokenize(sample.lower())) - len(cleaned)} stop words")
print(f"Kept {len(cleaned)} meaningful tokens")

# visualise most common stop words removed
removed = [t.lower() for t in word_tokenize(sample.lower()) 
           if t.lower() in stop_words]
print(f"Stop words removed: {removed}")

       STOP WORDS & TEXT CLEANING
Total English stop words: 198
Examples: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]

ORIGINAL:  'The movie was not very good but the acting was absolutely brilliant!'
CLEANED:    ['movie', 'good', 'acting', 'absolutely', 'brilliant']

Removed 8 stop words
Kept 5 meaningful tokens
Stop words removed: ['the', 'was', 'not', 'very', 'but', 'the', 'was']


## 4. Stemming & Lemmatisation

In [4]:
print("=" * 55)
print("       STEMMING & LEMMATISATION")
print("=" * 55)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words_to_test = [
    'running', 'runs', 'ran', 'runner',
    'better', 'good', 'best',
    'studies', 'studying', 'studied',
    'happily', 'happiness', 'happy',
    'caring', 'cared', 'cares'
]

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatised':<15}")
print("-" * 45)
for word in words_to_test:
    stemmed = stemmer.stem(word)
    lemmed  = lemmatizer.lemmatize(word, pos='v')
    print(f"{word:<15} {stemmed:<15} {lemmed:<15}")

print(f"""
KEY DIFFERENCES:
  Stemming     → chops off suffixes (fast but crude)
                 'studies' → 'studi' (not a real word!)
  Lemmatisation → finds dictionary root form (slower but accurate)
                 'studies' → 'study' (real word!)

WHEN TO USE:
  Stemming      → speed matters, search engines
  Lemmatisation → accuracy matters, NLP models
""")

       STEMMING & LEMMATISATION
Word            Stemmed         Lemmatised     
---------------------------------------------
running         run             run            
runs            run             run            
ran             ran             run            
runner          runner          runner         
better          better          better         
good            good            good           
best            best            best           
studies         studi           study          
studying        studi           study          
studied         studi           study          
happily         happili         happily        
happiness       happi           happiness      
happy           happi           happy          
caring          care            care           
cared           care            care           
cares           care            care           

KEY DIFFERENCES:
  Stemming     → chops off suffixes (fast but crude)
                 'studies' → 'studi

## 5. TF-IDF Vectorisation

In [5]:
print("=" * 55)
print("       TF-IDF VECTORISATION")
print("=" * 55)
print("""
TF-IDF = Term Frequency × Inverse Document Frequency

TF  (Term Frequency)     = how often word appears in THIS document
IDF (Inverse Doc Freq)   = how rare the word is across ALL documents
                           rare words get higher scores

INTUITION:
  'the'  → appears everywhere → low IDF → low TF-IDF score
  'brilliant' → rare → high IDF → high TF-IDF score
  Common words get penalised, rare meaningful words rewarded!
""")

corpus = [
    "the movie was absolutely brilliant and wonderful",
    "terrible film awful acting complete waste of time",
    "great story loved the characters and the plot",
    "boring movie bad acting would not recommend",
    "fantastic film brilliant director loved every scene",
    "worst movie ever seen terrible and disappointing",
]

vectorizer = TfidfVectorizer(max_features=20, stop_words='english')
X_tfidf = vectorizer.fit_transform(corpus)

print(f"Corpus: {len(corpus)} documents")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"TF-IDF matrix shape: {X_tfidf.shape}")

# show top words per document
feature_names = vectorizer.get_feature_names_out()
print(f"\nTop 3 TF-IDF words per document:")
for i, doc in enumerate(corpus):
    scores = X_tfidf[i].toarray()[0]
    top3   = sorted(zip(feature_names, scores),
                    key=lambda x: x[1], reverse=True)[:3]
    print(f"  Doc {i+1}: {[(w,round(s,3)) for w,s in top3]}")
    print(f"         '{doc[:45]}'")

       TF-IDF VECTORISATION

TF-IDF = Term Frequency × Inverse Document Frequency

TF  (Term Frequency)     = how often word appears in THIS document
IDF (Inverse Doc Freq)   = how rare the word is across ALL documents
                           rare words get higher scores

INTUITION:
  'the'  → appears everywhere → low IDF → low TF-IDF score
  'brilliant' → rare → high IDF → high TF-IDF score
  Common words get penalised, rare meaningful words rewarded!

Corpus: 6 documents
Vocabulary size: 20
TF-IDF matrix shape: (6, 20)

Top 3 TF-IDF words per document:
  Doc 1: [('absolutely', np.float64(0.682)), ('brilliant', np.float64(0.559)), ('movie', np.float64(0.472))]
         'the movie was absolutely brilliant and wonder'
  Doc 2: [('awful', np.float64(0.499)), ('complete', np.float64(0.499)), ('acting', np.float64(0.409))]
         'terrible film awful acting complete waste of '
  Doc 3: [('characters', np.float64(0.522)), ('great', np.float64(0.522)), ('plot', np.float64(0.522))]
     

## 6. Text Classification Pipeline

In [6]:
print("=" * 55)
print("       TEXT CLASSIFICATION PIPELINE")
print("=" * 55)

# larger dataset — movie reviews
reviews = [
    "absolutely brilliant film loved every minute",
    "wonderful movie fantastic acting great story",
    "amazing film beautiful cinematography loved it",
    "excellent movie highly recommend must watch",
    "brilliant director fantastic cast loved the film",
    "great movie wonderful story brilliant acting",
    "loved this film absolutely amazing experience",
    "fantastic movie great characters excellent plot",
    "terrible movie awful acting complete waste",
    "horrible film boring story would not recommend",
    "worst movie ever seen absolutely dreadful",
    "awful film terrible acting waste of time",
    "disappointing movie bad story poor acting",
    "dreadful film horrible waste of money",
    "boring terrible movie not worth watching",
    "awful waste of time terrible film bad acting",
]
labels = [1]*8 + [0]*8  # 1=positive, 0=negative

# pipeline: TF-IDF + Logistic Regression
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=100,
                              stop_words='english')
X = vectorizer.fit_transform(reviews)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# train two classifiers
models_clf = {
    'Naive Bayes':        MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)
for name, clf in models_clf.items():
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{name:<25} {acc*100:>9.1f}%")

# test on new reviews
print(f"\nPREDICTIONS ON NEW REVIEWS:")
lr = models_clf['Logistic Regression']
test_reviews = [
    "absolutely loved this brilliant film",
    "terrible waste of time awful movie",
    "great acting wonderful story enjoyed it",
    "boring dreadful film not worth watching",
]
for review in test_reviews:
    vec   = vectorizer.transform([review])
    pred  = lr.predict(vec)[0]
    prob  = lr.predict_proba(vec)[0].max()
    label = "POSITIVE 😊" if pred == 1 else "NEGATIVE 😞"
    print(f"  {label} ({prob:.2f}) — '{review}'")

       TEXT CLASSIFICATION PIPELINE
Model                       Accuracy
-------------------------------------
Naive Bayes                   100.0%
Logistic Regression           100.0%

PREDICTIONS ON NEW REVIEWS:
  POSITIVE 😊 (0.59) — 'absolutely loved this brilliant film'
  NEGATIVE 😞 (0.59) — 'terrible waste of time awful movie'
  POSITIVE 😊 (0.52) — 'great acting wonderful story enjoyed it'
  NEGATIVE 😞 (0.57) — 'boring dreadful film not worth watching'


## 7. Key Takeaways

In [7]:
print("=" * 55)
print("       DAY 41 — KEY TAKEAWAYS")
print("=" * 55)
print("""
TOKENISATION:
  ✅ Sentence tokenisation → splits text into sentences
  ✅ Word tokenisation     → splits into words + punctuation
  ✅ NLTK smarter than split() — handles punctuation correctly

STOP WORDS:
  ✅ 198 English stop words — the, is, and, was...
  ✅ Remove to keep only meaningful tokens
  ✅ WARNING: removing 'not' can flip sentiment!

STEMMING vs LEMMATISATION:
  ✅ Stemming      → chops suffix → fast but crude
                    'studies' → 'studi' (not real word)
  ✅ Lemmatisation → dictionary root → slow but accurate
                    'studies' → 'study' (real word)

TF-IDF:
  ✅ TF  = how often word appears in document
  ✅ IDF = how rare word is across all documents
  ✅ Rare meaningful words score highest
  ✅ Common words penalised automatically

TEXT CLASSIFICATION PIPELINE:
  ✅ Clean text → TF-IDF vectorise → train classifier
  ✅ Naive Bayes       → 100% accuracy
  ✅ Logistic Regression → 100% accuracy
  ✅ ngram_range=(1,2) → captures word pairs too

NLP PIPELINE SUMMARY:
  Raw text
    → Clean (lowercase, remove punctuation)
    → Tokenise
    → Remove stop words
    → Stem / Lemmatise
    → Vectorise (TF-IDF)
    → Train classifier
    → Predict!
""")

       DAY 41 — KEY TAKEAWAYS

TOKENISATION:
  ✅ Sentence tokenisation → splits text into sentences
  ✅ Word tokenisation     → splits into words + punctuation
  ✅ NLTK smarter than split() — handles punctuation correctly

STOP WORDS:
  ✅ 198 English stop words — the, is, and, was...
  ✅ Remove to keep only meaningful tokens
  ✅ WARNING: removing 'not' can flip sentiment!

STEMMING vs LEMMATISATION:
  ✅ Stemming      → chops suffix → fast but crude
                    'studies' → 'studi' (not real word)
  ✅ Lemmatisation → dictionary root → slow but accurate
                    'studies' → 'study' (real word)

TF-IDF:
  ✅ TF  = how often word appears in document
  ✅ IDF = how rare word is across all documents
  ✅ Rare meaningful words score highest
  ✅ Common words penalised automatically

TEXT CLASSIFICATION PIPELINE:
  ✅ Clean text → TF-IDF vectorise → train classifier
  ✅ Naive Bayes       → 100% accuracy
  ✅ Logistic Regression → 100% accuracy
  ✅ ngram_range=(1,2) → captures word 